In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import logging
import numpy as np
import joblib
from pathlib import Path

from config import Ablation, Species
from models import train_and_explain   

In [ ]:
# Configuration
SPECIES: list[Species] = ["spruce", "pine", "beech", "oak"]
MODEL_TYPES = ["gbdt", "lasso"]
ABLATIONS: list[Ablation] = ["all", "no-defoliation", "plot-level-only", "tree-level-only"]

GROUP_BY = "tree_id"
USE_TEMPORAL_CV = True

if GROUP_BY == "plot_id":
    USE_TEMPORAL_CV = False
else:
    USE_TEMPORAL_CV = USE_TEMPORAL_CV

CACHE_DIR = Path("./cache")
CACHE_DIR.mkdir(exist_ok=True)

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

if USE_TEMPORAL_CV:
    temporal_label = "with_temp_blocking"
else:
    temporal_label = "without_temp_blocking"

In [ ]:
def get_experiment(species: Species, model_type: str, ablation: Ablation):
    logging.info(f"Running experiment: {species} {model_type} {ablation}")
    exp = train_and_explain(
        species=species,
        model_type=model_type,      # type: ignore
        ablation=ablation,
        group_by=GROUP_BY,
        use_temporal_cv=USE_TEMPORAL_CV,
    )
    return exp


results = {mt: {ab: {} for ab in ABLATIONS} for mt in MODEL_TYPES}

for species in SPECIES:
    for model_type in MODEL_TYPES:
        for ablation in ABLATIONS:
            exp = get_experiment(species, model_type, ablation)
            test_r2_scores = [perf["test_r2"] for perf in exp.performances]
            mean_r2 = np.mean(test_r2_scores)
            std_r2 = np.std(test_r2_scores, ddof=1)
            test_rmse_scores = [perf["test_rmse"] for perf in exp.performances]
            mean_rmse = np.mean(test_rmse_scores)
            total_n_test = sum(perf["n_test"] for perf in exp.performances)
            results[model_type][ablation][species] = (mean_r2, std_r2, total_n_test, mean_rmse)


def fmt(mean: float, std: float) -> str:
    return f"{mean:.2f} ± {std:.2f}"


In [ ]:
ABLATION_NAMES = {
            "all": "all features",
            "no-defoliation": "no defoliation",
            "plot-level-only": "plot-level features only",
            "tree-level-only": "tree-level features only",
        }


COL_MODEL = 30         
COL_SPECIES = 18        
COL_WMEAN = 20         
COL_RMSE = 15           

# Header
header = (
    f"{'Model / Features':<{COL_MODEL}}"
    + "".join(f"{sp.capitalize():<{COL_SPECIES}}" for sp in SPECIES)
    + f"{'Weighted Mean R2':<{COL_WMEAN}}"
    + f"{'Weighted RMSE':<{COL_RMSE}}"
)
print("\n" + "=" * len(header))
print(header)
print("-" * len(header))


for model_type in MODEL_TYPES:
    for ablation in ABLATIONS:
        print(f"{model_type.upper():<{COL_MODEL}}")
        feature_desc = f"({ABLATION_NAMES[ablation]})"
        species_vals = []
        weights = []
        for species in SPECIES:
            mean_r2, std_r2, n_test, mean_rmse = results[model_type][ablation][species]
            species_vals.append(f"{mean_r2:.2f} ± {std_r2:.2f}")
            weights.append(n_test)
        means = [results[model_type][ablation][sp][0] for sp in SPECIES]
        rmses = [results[model_type][ablation][sp][3] for sp in SPECIES]
        weighted_mean = np.average(means, weights=weights)
        weighted_rmse = np.average(rmses, weights=weights)
       
        line = f"{feature_desc:<{COL_MODEL}}"
        line += "".join(f"{val:<{COL_SPECIES}}" for val in species_vals)
       
        line += f"{weighted_mean:<{COL_WMEAN}.2f}"
        line += f"{weighted_rmse:<{COL_RMSE}.2f}"
        print(line)

print("=" * len(header))